# Data assimilation in the latent space: Particle filter

## Experiment output folder

In [1]:
from pathlib import Path

# Define the subfolder path for outputs
output_dir = Path("output/ETNA2018")

## Open prior and observation datasets

In [2]:
import xarray as xr
import numpy as np

fname_obs = "data/20181224_1800_Meteosat-11_Etna_VPRoutput.nc"

prior = xr.open_dataset("output/ETNA2018/prior.nc")
obs = xr.open_dataset(fname_obs)

In [11]:
nens = prior.sizes['ens']
latent_dim = prior.sizes['latent_dim']
print(f"Ensemble size: {nens} --- Latent space size: {latent_dim}")

Ensemble size: 4096 --- Latent space size: 32


# Get observations

In [4]:
valid = (obs["plume_mask"].values > 0) & (obs["mass"].values > 0)
y_obs = obs["mass"].values[valid].astype(np.float64)
lat_obs = obs["latitude"].values[valid].astype(np.float64)
lon_obs = obs["longitude"].values[valid].astype(np.float64)

In [12]:
nobs = len(y_obs)
print(f"Number of observations: {nobs}")

Number of observations: 2678


In [6]:
# Required for interpolations
lat_points = xr.DataArray(lat_obs, dims="obs")
lon_points = xr.DataArray(lon_obs, dims="obs")

## Get the prior ensemble

In [7]:
Z = prior["z"].values.astype(np.float64)
X = prior["samples"]

Y_xr = X.interp(
    lat=lat_points,
    lon=lon_points,
    method="linear"
)
Y = Y_xr.values.astype(np.float64)

print("Y shape:", Y.shape)

Y shape: (4096, 2678)


In [8]:
print("NaNs in Y:", np.isnan(Y).sum())

NaNs in Y: 0


## Observation-error covariance

In [10]:
# 50% error observation is assumed with a minimim of
# 0.1 g/m2 (typical satellite detection limit)
obs_error_fraction = 0.50
obs_error_std = obs_error_fraction * y_obs
obs_error_std[obs_error_std<0.1] = 0.1 # Apply a minimum error (satellite detection limit)

print("error std range:", obs_error_std.min(), obs_error_std.max())

error std range: 0.1 1.02644674054899


## Check dimensions

In [13]:
assert Z.shape == (nens, latent_dim), "Wrong dimensions for Z"
assert Y.shape == (nens, nobs), "Wrong dimensions for Y"
assert y_obs.shape == (nobs,), "Wrong dimensions for y_obs"

## Normalized particle-filter weights

In [14]:
def particle_weights(Y, y_obs, sigma_obs):
    """
    Compute normalized particle-filter weights.

    Parameters
    ----------
    Y : ndarray, shape (nens, nobs)
        Observation-space ensemble.
    y_obs : ndarray, shape (nobs,)
        Observations.
    sigma_obs : ndarray or float
        Observation standard deviation(s).

    Returns
    -------
    weights : ndarray, shape (nens,)
        Normalized particle weights.
    log_weights : ndarray, shape (nens,)
        Unnormalized log-likelihoods.
    """

    # Innovation for every particle
    innovation = Y - y_obs[None, :]

    # Mahalanobis distance for diagonal R
    log_weights = -0.5 * np.sum(
        (innovation / sigma_obs) ** 2,
        axis=1
    )

    # Numerically stable normalization
    log_weights -= np.max(log_weights)
    weights = np.exp(log_weights)
    weights /= np.sum(weights)

    return weights, log_weights

In [15]:
weights, log_weights = particle_weights(Y,y_obs,obs_error_std)

In [16]:
print("min weight:", weights.min())
print("max weight:", weights.max())
print("sum weights:", weights.sum())

min weight: 0.0
max weight: 0.775918996339853
sum weights: 1.0


In [17]:
N_eff = 1.0 / np.sum(weights**2)

print("Effective sample size:", N_eff)
print("N_eff / nens:", N_eff / len(weights))

Effective sample size: 1.5363288642990194
N_eff / nens: 0.0003750802891355028


## Systematic resampling

In [19]:
def systematic_resample(weights, rng=None):
    """
    Systematic resampling.

    Parameters
    ----------
    weights : ndarray, shape (nens,)
        Normalized particle weights. Must sum to 1.
    rng : np.random.Generator, optional
        Random number generator.

    Returns
    -------
    indices : ndarray, shape (nens,)
        Indices of resampled particles.
    """
    if rng is None:
        rng = np.random.default_rng()

    weights = np.asarray(weights)
    n = len(weights)

    # Cumulative distribution
    cdf = np.cumsum(weights)
    cdf[-1] = 1.0  # avoid numerical issues

    # One random offset, then equally spaced points
    u0 = rng.uniform(0.0, 1.0 / n)
    u = u0 + np.arange(n) / n

    # Find corresponding particles
    indices = np.searchsorted(cdf, u)

    return indices

In [20]:
rng = np.random.default_rng(42)

indices = systematic_resample(weights, rng)

Z_resampled = Z[indices]
X_resampled = X[indices]
Y_resampled = Y[indices]

In [21]:
unique, counts = np.unique(indices, return_counts=True)

print("Number of unique particles:", len(unique))
print("Fraction unique:", len(unique) / len(Z))

order = np.argsort(counts)[::-1]

for i in order[:10]:
    print(unique[i], counts[i], counts[i] / len(Z))

Number of unique particles: 4
Fraction unique: 0.0009765625
336 3178 0.77587890625
2078 905 0.220947265625
2882 10 0.00244140625
3590 3 0.000732421875


## Weighted mean and covariance

In [22]:
def weighted_mean_covariance(Z, weights):
    """
    Weighted mean and covariance of an ensemble.

    Parameters
    ----------
    Z : ndarray, shape (nens, latent_dim)
        Prior latent ensemble.
    weights : ndarray, shape (nens,)
        Normalized particle weights.

    Returns
    -------
    mean : ndarray, shape (latent_dim,)
    cov : ndarray, shape (latent_dim, latent_dim)
    """

    Z = np.asarray(Z)
    weights = np.asarray(weights)

    # Weighted mean
    mean = np.sum(weights[:, None] * Z, axis=0)

    # Anomalies
    A = Z - mean

    # Weighted covariance
    cov = (A * weights[:, None]).T @ A

    return mean, cov

In [23]:
z_mean, z_cov = weighted_mean_covariance(Z, weights)

print("mean:", z_mean)
print("cov shape:", z_cov.shape)

mean: [-0.52319653  0.34006405 -1.18840817 -0.53779076 -0.2349062   0.43741097
 -0.3620888   0.71252338  0.61405478 -0.11880565  0.32380948  0.04688141
  0.01123177 -0.39790613 -0.96340583  0.35137808 -0.92869822  1.15096293
  0.9928178  -0.375776   -0.35204038  0.07574427 -0.85842786  0.10374619
 -0.20036297  0.60550894 -0.67630537 -0.2582777   0.109184    0.05133648
 -0.75445796 -2.00088882]
cov shape: (32, 32)


In [24]:
# Check the covariance
eigvals = np.linalg.eigvalsh(z_cov)

print("Eigenvalues:")
print(eigvals)

print("min eigenvalue:", eigvals.min())
print("max eigenvalue:", eigvals.max())

Eigenvalues:
[-6.60523797e-16 -4.83301134e-16 -2.30389710e-16 -1.52974922e-16
 -1.32418885e-16 -8.00160299e-17 -7.09807072e-17 -5.97884542e-17
 -3.76612746e-17 -2.80240982e-17 -1.72721877e-17 -5.05927815e-18
 -3.11600845e-18  2.13471201e-19  8.55398982e-18  1.81522250e-17
  2.53306785e-17  3.96741879e-17  6.41557980e-17  9.03531798e-17
  1.17829083e-16  2.04532069e-16  2.67121865e-16  3.69970829e-16
  1.05030726e-15  2.11846207e-13  1.89390989e-11  9.88030889e-08
  2.14624623e-06  1.81798115e-02  7.39446364e-02  1.32297655e+01]
min eigenvalue: -6.60523796869307e-16
max eigenvalue: 13.229765525611791


In [25]:
prior_mean = np.mean(Z, axis=0)
prior_cov = np.cov(Z, rowvar=False)

print("Prior std:")
print(np.sqrt(np.diag(prior_cov)))

print("Weighted std:")
print(np.sqrt(np.diag(z_cov)))

Prior std:
[1.02027302 0.98067448 0.89709747 0.98580598 1.04842622 1.03886114
 1.00576963 0.8931671  0.96477183 0.91847079 1.00133909 0.93551039
 0.8719525  1.01062976 1.05375082 1.13682734 1.05964953 0.9194666
 1.04895163 1.14465355 1.06048471 1.02570579 0.99032141 1.04858926
 0.89151934 1.03522836 1.00237242 1.03416664 0.90713998 1.01456366
 0.90591765 1.11573742]
Weighted std:
[1.92159036 0.87405793 0.41802338 0.69197663 0.27366645 0.31768546
 0.23907981 0.12383956 0.88320367 0.39817592 0.5059498  0.72777166
 0.15545675 0.39493282 0.42703507 0.03471047 0.97406925 0.2942332
 0.14808141 0.38395942 0.20792562 0.20893969 0.7721669  0.1673944
 0.59194442 0.68655011 0.75629996 0.74037988 0.57813972 0.2387702
 1.06430663 0.70270214]


## Kernel perturbation

In [47]:
regularization_factor = 0.2

Sigma_reg = prior_cov * regularization_factor**2

# Ensure exact symmetry
Sigma_reg = 0.5 * (Sigma_reg + Sigma_reg.T)

# Cholesky factor
jitter = 1e-10 * np.trace(Sigma_reg) / latent_dim

L = np.linalg.cholesky(Sigma_reg + jitter * np.eye(latent_dim))

# Generate correlated perturbations
noise = np.random.randn(nens, latent_dim)
perturbation = noise @ L.T

# Regularized posterior
Z_analysis = Z_resampled + perturbation

In [36]:
print("Z_resampled std:", Z_resampled.std(axis=0))
print("Z_posterior std:", Z_analysis.std(axis=0))

Z_resampled std: [1.92138867 0.8739895  0.41805192 0.69189968 0.2736833  0.3177689
 0.23921502 0.12386008 0.88333384 0.39818583 0.50605845 0.72769367
 0.15565426 0.39490921 0.42709192 0.03475367 0.97395483 0.29485867
 0.1480998  0.38394762 0.20834649 0.20894871 0.77222426 0.16761509
 0.59201697 0.68671901 0.75626757 0.74035381 0.57821729 0.2388683
 1.06418443 0.70302975]
Z_posterior std: [1.92878534 0.87733837 0.41981859 0.69448347 0.27471264 0.31913347
 0.23991039 0.12428009 0.88689669 0.39953185 0.50797026 0.73052146
 0.15637216 0.39629956 0.42892347 0.03486206 0.97766008 0.29623329
 0.14868081 0.38542483 0.20928607 0.20982166 0.77498979 0.16821515
 0.59444626 0.68954275 0.75914946 0.74294424 0.58055341 0.23974139
 1.06817968 0.70572713]


## Summary

In [49]:
Pz_prior = np.cov(Z, rowvar=False)
Pz_analysis = np.cov(Z_analysis, rowvar=False)

In [50]:
variance_prior = np.trace(Pz_prior)
variance_analysis = np.trace(Pz_analysis)

print("Total prior variance:   ", variance_prior)
print("Total analysis variance:", variance_analysis)
print("Variance ratio:",variance_analysis / variance_prior)

Total prior variance:    32.09803868076218
Total analysis variance: 14.632711740548359
Variance ratio: 0.4558755719027285


## Save posterior

In [48]:
posterior_latent = xr.Dataset(
    data_vars={
        "z": (
            ("ens", "latent_dim"),
            Z_analysis.astype(np.float32)
        ),
        "z_mean": (
            ("latent_dim",),
            Z_analysis.mean(axis=0).astype(np.float32)
        ),
    },
    coords={
        "ens": np.arange(N_ens),
        "latent_dim": np.arange(latent_dim),
    },
)

posterior_latent.to_netcdf(output_dir / "posterior-latent-pf.nc")

In [39]:
Z_analysis.mean(axis=0)

array([-0.52238811,  0.3404615 , -1.18859579, -0.53755826, -0.23484871,
        0.43751182, -0.36204741,  0.71246996,  0.61353587, -0.1186754 ,
        0.3237348 ,  0.04660059,  0.0113613 , -0.39804019, -0.96319214,
        0.35138245, -0.92832479,  1.15067666,  0.99273301, -0.37596508,
       -0.35183863,  0.07566101, -0.85857227,  0.10386867, -0.20002435,
        0.60508139, -0.67594275, -0.25807197,  0.10884784,  0.05148067,
       -0.75404352, -2.0004278 ])